# Fine tuning
In this notebook, an opensource sentencetransformers embedding model is finetuned on the synthetically generated dataset.

## Load pretrained model

In [1]:
!nvidia-smi

Wed May  8 01:59:14 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla T4                       Off | 00000000:00:04.0 Off |                    0 |
| N/A   54C    P8              10W /  70W |      0MiB / 15360MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [2]:
!pip install sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 1.5 MB/s eta 0:00:00


In [3]:
from sentence_transformers import SentenceTransformer

In [4]:
import torch

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Staring with BAAI/bge-small-en

In [5]:
# Load pretrained model and move it to GPU
model_id = "BAAI/bge-small-en"
model = SentenceTransformer(model_id)
model.to(device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/90.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [6]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

## Define dataloader

In [7]:
import json

from torch.utils.data import DataLoader
from sentence_transformers import InputExample

In [8]:
TRAIN_DATASET_PATH = '/kaggle/input/split-dataset-1500/train_dataset.json'
VAL_DATASET_PATH = '/kaggle/input/split-dataset-1500/val_dataset.json'

# We use a very small batchsize to run this toy example on a local machine. 
# This should typically be much larger. 
BATCH_SIZE = 16

In [9]:
with open(TRAIN_DATASET_PATH, 'r') as f:
    train_dataset = json.load(f)

with open(VAL_DATASET_PATH, 'r') as f:
    val_dataset = json.load(f)

In [10]:
dataset = train_dataset

corpus = dataset['corpus']
queries = dataset['queries']
relevant_docs = dataset['relevant_docs']

examples = []
for query_id, query in queries.items():
    chunk_id = relevant_docs[query_id][0]
    text = corpus[chunk_id]
    example = InputExample(texts=[query, text])
    examples.append(example)

In [11]:
loader = DataLoader(
    examples, batch_size=BATCH_SIZE
)

## Define loss
**MultipleNegativesRankingLoss** is a great loss function if you only have positive pairs, for example, only pairs of similar texts like pairs of paraphrases, pairs of duplicate questions, pairs of (query, response), or pairs of (source_language, target_language).

This loss function works great to train embeddings for retrieval setups where you have positive pairs (e.g. (query, relevant_doc)) as it will sample in each batch n-1 negative docs randomly.

The performance usually increases with increasing batch sizes.

For more detals, see:

docs
[paper](

In [12]:
from sentence_transformers import losses

In [13]:
loss = losses.MultipleNegativesRankingLoss(model)

## Define evaluator
We setup an evaluator with our val split of the dataset to monitor how well the embedding model is performing during training.

In [14]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

In [15]:
dataset = val_dataset

corpus = dataset['corpus']
queries = dataset['queries']
relevant_docs = dataset['relevant_docs']

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_docs)

## Run training
The training loop is very straight forward to steup thanks to sentencetransformers' high-level model training API. All we need to do is plugging in the data loader, loss function, and evaluator that we defined in the previous cells (along with a couple of additional minor settings).

In [16]:
# We train the model for very few epochs in this toy example.
# This should typically be higher for better performance.
EPOCHS = 40

In [17]:
warmup_steps = int(len(loader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path='bge-small_finetuned',
    show_progress_bar=True,
    evaluator=evaluator, 
    evaluation_steps=50
)

Epoch:   0%|          | 0/40 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

## Finetuning with BAAI/bge-large-en-v1.5

In [18]:
model_id = "BAAI/bge-large-en-v1.5"
model = SentenceTransformer(model_id)
model.to(device)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': True}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)

In [19]:
dataset = train_dataset

corpus = dataset['corpus']
queries = dataset['queries']
relevant_docs = dataset['relevant_docs']

examples = []
for query_id, query in queries.items():
    chunk_id = relevant_docs[query_id][0]
    text = corpus[chunk_id]
    example = InputExample(texts=[query, text])
    examples.append(example)

In [20]:
loader = DataLoader(
    examples, batch_size=BATCH_SIZE
)

In [21]:
loss = losses.MultipleNegativesRankingLoss(model)

In [22]:
dataset = val_dataset

corpus = dataset['corpus']
queries = dataset['queries']
relevant_docs = dataset['relevant_docs']

evaluator = InformationRetrievalEvaluator(queries, corpus, relevant_docs)

In [23]:
warmup_steps = int(len(loader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path='bge-large_finetuned',
    show_progress_bar=True,
    evaluator=evaluator, 
    evaluation_steps=50
)

Epoch:   0%|          | 0/40 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

...

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]

Iteration:   0%|          | 0/66 [00:00<?, ?it/s]